# 41 — Run Blind-A inference + package CodaBench zip

**Use case** (Option B refactor + corrected Blind-A timeline): Blind-A is
**alive through 2026-06-23**. We submit weekly to harvest real Gemini-judge
scores against our own outputs — those scores become future calibration
targets for the distilled judge AND validate the Component-B refactor's
direction.

**Per-week budget**: 3/week per `scripts/validate_prediction.py:172` (the
existing weekly cap). Spend each on a distinctly-different config (one-axis
diff per `feedback_experiment_ablation_discipline`).

## Sequence

1. Resolve merged repo (W6-pilot > W6-full > W5 > W4) from gate_result.json.
2. Patch `config/301-pilot-blindset-A.yaml` with the merged repo.
3. Run `run_inference_blindset.py --tid 301-pilot-blindset-A --eval_dataset blindset_A`.
4. Validate + package with `--split blindA`.
5. Stage zip to Drive.

In [ ]:
# 1) GPU check.
!nvidia-smi | head -10

In [ ]:
# 2) Clone fresh-model branch.
BRANCH = 'fresh-model'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

In [ ]:
# 2b) Drive + caches + HF auth.
import os, shutil
from google.colab import drive, userdata
try: drive.mount('/content/drive')
except Exception:
    try: drive.flush_and_unmount()
    except Exception: pass
    drive.mount('/content/drive', force_remount=True)

DRIVE_BASE = '/content/drive/MyDrive/recsys2026-cache'
for d in [f'{DRIVE_BASE}/hf_datasets', f'{DRIVE_BASE}/experiments_cache', f'{DRIVE_BASE}/blindset_runs']:
    os.makedirs(d, exist_ok=True)

os.environ['HF_DATASETS_CACHE'] = f'{DRIVE_BASE}/hf_datasets'
%env HF_DATASETS_CACHE={DRIVE_BASE}/hf_datasets

EXPECTED_CACHE = '/content/recsys2026/music-crs-baselines/experiments/cache'
os.makedirs(os.path.dirname(EXPECTED_CACHE), exist_ok=True)
if os.path.exists(EXPECTED_CACHE) and not os.path.islink(EXPECTED_CACHE):
    shutil.rmtree(EXPECTED_CACHE)
if not os.path.islink(EXPECTED_CACHE):
    os.symlink(f'{DRIVE_BASE}/experiments_cache', EXPECTED_CACHE)

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['HUGGINGFACE_HUB_TOKEN'] = os.environ['HF_TOKEN']
print('✓ ready')

In [ ]:
# 3) Resolve merged repo: prefer pilot > full W6 > W7 > W5 > W4.
import json
from pathlib import Path

def latest(runs_dir):
    p = Path(runs_dir)
    if not p.exists(): return None
    cands = list(p.rglob('gate_result.json'))
    if not cands: return None
    return json.load(open(max(cands, key=lambda x: x.stat().st_mtime)))

PILOT = latest(f'{DRIVE_BASE}/grpo_pilot_runs')
W6 = latest(f'{DRIVE_BASE}/grpo_runs')
W7 = latest(f'{DRIVE_BASE}/grpo_final_runs')
W5 = latest(f'{DRIVE_BASE}/sdpo_runs')
W4 = latest(f'{DRIVE_BASE}/kto_runs')

candidates = [
    ('W7-final', W7),
    ('W6-full', W6),
    ('W6-pilot', PILOT),
    ('W5-sdpo', W5),
    ('W4-kto', W4),
]
MERGED = None; STAGE = None
for name, gate in candidates:
    if gate and gate.get('merged_hub_model'):
        MERGED = gate['merged_hub_model']; STAGE = name
        print(f'  ✓ found {name}: {MERGED}')
        break
    elif gate:
        print(f'  - {name}: gate exists but no merged_hub_model')

if MERGED is None:
    raise SystemExit('❌ No merged repo from any stage. Run W4-W7 first.')

print(f'\nUsing for Blind-A: {STAGE} → {MERGED}')

In [ ]:
# 4) Patch config 301 with the resolved merged repo + disable vLLM.
#
# 2026-05-14 fix: the inference pipeline shares the GPU between retrieval models
# (Llama-3.2-1B + 2× dense embedders + ProRank cross-encoder) AND the responder
# LLM. vLLM pre-allocates 70% of GPU memory at startup, which collides with
# the retrieval stack already loaded → "Free memory ... less than desired GPU
# memory utilization" startup error. HF transformers backend allocates on
# demand and shares cleanly. ~3-4× slower per-turn but reliable.
import re
CONFIG_301 = '/content/recsys2026/music-crs-baselines/config/301-pilot-blindset-A.yaml'

with open(CONFIG_301, encoding='utf-8') as f: txt = f.read()
new = re.sub(r'^lm_type:\s*"[^"]*"', f'lm_type: "{MERGED}"', txt, flags=re.MULTILINE)
if new == txt:
    raise RuntimeError('failed to rewrite lm_type in config 301')

# Also force use_vllm: false to avoid the GPU-sharing OOM with retrieval stack.
new2 = re.sub(r'^use_vllm:\s*true', 'use_vllm: false', new, flags=re.MULTILINE)
if new2 == new:
    print('  (use_vllm already false or absent — leaving config as-is)')
else:
    print('  ✓ patched use_vllm: true → false (HF transformers backend)')

with open(CONFIG_301, 'w', encoding='utf-8') as f: f.write(new2)
!grep -E '^(lm_type|use_vllm):' /content/recsys2026/music-crs-baselines/config/301-pilot-blindset-A.yaml


In [ ]:
# 5) Install deps + pytest pre-flight.
!pip install -q --upgrade transformers datasets 'pandas<3.0' tqdm omegaconf pyyaml
!pip install -q --upgrade 'trl>=0.12.0' 'peft>=0.13.0' 'torchao>=0.16.0' bm25s

!cd /content/recsys2026 && python -m pytest \
    tests/test_reward_fns.py \
    tests/test_prediction_validator.py \
    tests/test_train_distilled_judge.py \
    tests/test_build_grpo_dataset.py \
    -q

In [ ]:
# 6) Run inference on Blind-A.
BLIND_A_OUT = '/content/recsys2026/music-crs-baselines/exp/inference/blindset_A/301-pilot-blindset-A.json'

if not Path(BLIND_A_OUT).exists():
    !cd /content/recsys2026/music-crs-baselines && python run_inference_blindset.py \
        --tid 301-pilot-blindset-A --eval_dataset blindset_A \
        --batch_size 8 --device cuda --attn_implementation sdpa
else:
    print(f'reusing existing {BLIND_A_OUT}')

import json
with open(BLIND_A_OUT, encoding='utf-8') as f: blind_a = json.load(f)
print(f'\nBlind-A rows produced: {len(blind_a)}')

In [ ]:
# 7) Validate schema + package CodaBench zip (split=blindA).
from datetime import date
import sys
sys.path.insert(0, '/content/recsys2026/scripts')
from validate_prediction import load_prediction, validate_schema, package_zip

ZIP_PATH = f'/content/recsys2026/data/submissions/blindset_A_{date.today().isoformat()}_301.zip'
os.makedirs(os.path.dirname(ZIP_PATH), exist_ok=True)

predictions = load_prediction(BLIND_A_OUT)
errors = validate_schema(predictions, 'blindA')
if errors:
    print('❌ Schema validation FAILED:')
    for e in errors[:20]: print(f'  - {e}')
    raise SystemExit('Refusing to package — fix inference output and rerun.')
print(f'✓ schema OK ({len(predictions)} rows for blindA — expected 80)')

out_zip = package_zip(BLIND_A_OUT, ZIP_PATH)
print(f'✓ packaged → {out_zip}')

# Drive copy.
import shutil
drive_zip = f'{DRIVE_BASE}/blindset_runs/blindset_A_{date.today().isoformat()}_301.zip'
shutil.copy(ZIP_PATH, drive_zip)
print(f'✓ Drive copy → {drive_zip}')

# Verify zip layout.
import zipfile
with zipfile.ZipFile(ZIP_PATH) as zf: members = zf.namelist()
assert members == ['prediction.json'], f'wrong layout: {members}'
print(f'✓ zip contains exactly [prediction.json]')

print(f'\n📤  UPLOAD this zip to CodaBench (Blind-A).')
print(f'    Source stage: {STAGE}')
print(f'    Source model: {MERGED}')
print(f'    File:         {ZIP_PATH}')

## After upload

**On a Gemini-aligned score back from CodaBench:**
1. Append a row to `documents/submissions_log.md` with the date, source stage, leaderboard score, and Gemini judge breakdown if available. The next blind submission's `validate_prediction --check-budget` reads this log.
2. **Use the Gemini score to update the distilled judge** — append (context, response, gemini_score) tuples to `data/reward_calibration_anchors.parquet` and re-train via `scripts/train_distilled_judge.py --mode train`. Each Blind-A round adds ~80 real Gemini-anchored points; over 6 weekly submissions that's ~480 anchors, which materially improves the cross-encoder's calibration on production-scale outputs.

**On a poor Gemini score (vs the W4-only baseline):**
- Diagnosis: the distilled judge may be over-confident about responses Gemini doesn't actually like. Either (a) drop `W_JUDGE` 0.30 → 0.15 in `reward_fns.py` and re-pilot, or (b) train the judge for more epochs / larger base model.